[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sunilmogadati/systems-in-production/blob/main/implementation/notebooks/Python_Advanced.ipynb)

# Advanced Python Patterns

Decorators, context managers, error handling, generators, type hints with Pydantic, and regular expressions — the patterns behind production AI and Data Engineering systems.

**Concept chapter:** [Advanced Python](https://github.com/sunilmogadati/systems-in-production/blob/main/playbooks/python/06_Advanced.md)

**Community:** [Delivery Momentum on Skool](https://www.skool.com/deliverymomentum)

## 1. Decorators

A decorator wraps a function to add behavior — logging, timing, retries, authentication. When you see `@timer`, `@retry`, or `@app.route`, you are looking at decorators.

In [ ]:
# WHAT: Understand what a decorator does — it wraps a function.
# WHY: Flask's @app.route, FastAPI's @app.get, pytest's @fixture,
# and sklearn's @deprecated all use this pattern.

import time
import functools

def timer(func):
    """Decorator that prints how long a function takes."""
    @functools.wraps(func)   # preserves the original function's name and docstring
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"  [{func.__name__}] completed in {elapsed:.4f}s")
        return result
    return wrapper

@timer
def train_model(n_samples: int) -> float:
    """Simulate model training."""
    # Simulate work
    total = sum(i * i for i in range(n_samples))
    return total / n_samples

result = train_model(1_000_000)
print(f"  Result: {result:.2f}")

# The decorator is equivalent to: train_model = timer(train_model)
# You Should See:
# [train_model] completed in 0.XXXXs
# Result: 333332833333.50 (or similar)

In [ ]:
# WHAT: Build a retry decorator with configurable attempts.
# WHY: API calls and database connections fail. Retrying with
# backoff is a production requirement — not optional.

import time
import functools
import random

def retry(max_attempts: int = 3, delay: float = 1.0):
    """Decorator factory — returns a decorator with config."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    if attempt == max_attempts:
                        raise
                    wait = delay * (2 ** (attempt - 1))  # exponential backoff
                    print(f"  Attempt {attempt} failed: {e}. Retrying in {wait}s...")
                    time.sleep(wait * 0.01)  # scaled down for demo
        return wrapper
    return decorator

@retry(max_attempts=3, delay=1.0)
def fetch_predictions(model_id: str) -> list:
    """Simulate an unreliable API call."""
    if random.random() < 0.6:
        raise ConnectionError("Model server unavailable")
    return [{"id": 1, "score": 0.95}]

random.seed(42)
try:
    result = fetch_predictions("bert-v2")
    print(f"  Success: {result}")
except ConnectionError as e:
    print(f"  All retries exhausted: {e}")

# You Should See:
# Attempt N failed: Model server unavailable. Retrying...
# (eventually Success or All retries exhausted)

## 2. Context Managers

The `with` statement guarantees cleanup — file closing, database disconnection, lock release. You already use it with `open()`. Here is how to build your own.

In [ ]:
# WHAT: Build a context manager for database connections.
# WHY: Unclosed connections leak memory and exhaust connection pools.
# The 'with' statement guarantees cleanup even if an error occurs.

class DatabaseConnection:
    """Simulated database connection with guaranteed cleanup."""
    
    def __init__(self, host: str, database: str):
        self.host = host
        self.database = database
        self.connected = False
    
    def __enter__(self):
        # Called when entering 'with' block
        print(f"  Connecting to {self.host}/{self.database}...")
        self.connected = True
        return self    # this becomes the 'as' variable
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        # Called when leaving 'with' block — ALWAYS, even on error
        print(f"  Closing connection to {self.host}/{self.database}")
        self.connected = False
        return False   # don't suppress exceptions
    
    def query(self, sql: str) -> list:
        if not self.connected:
            raise RuntimeError("Not connected")
        print(f"  Executing: {sql}")
        return [{"count": 42}]  # simulated result

# Use it — connection is GUARANTEED to close
with DatabaseConnection("prod-db", "analytics") as db:
    result = db.query("SELECT count(*) FROM calls")
    print(f"  Result: {result}")

print(f"  Connected after 'with': {db.connected}")

# You Should See:
#   Connecting to prod-db/analytics...
#   Executing: SELECT count(*) FROM calls
#   Result: [{'count': 42}]
#   Closing connection to prod-db/analytics
#   Connected after 'with': False

In [ ]:
# WHAT: Lightweight context manager with contextlib.
# WHY: When you just need setup/teardown without a full class.

from contextlib import contextmanager
import time

@contextmanager
def pipeline_stage(name: str):
    """Log timing for a pipeline stage."""
    print(f"  [{name}] Starting...")
    start = time.perf_counter()
    yield   # execution happens here
    elapsed = time.perf_counter() - start
    print(f"  [{name}] Completed in {elapsed:.4f}s")

with pipeline_stage("Extract"):
    data = list(range(100_000))

with pipeline_stage("Transform"):
    transformed = [x * 2 for x in data]

with pipeline_stage("Load"):
    total = sum(transformed)
    print(f"  Loaded {len(transformed):,} records (sum: {total:,})")

# You Should See:
#   [Extract] Starting...
#   [Extract] Completed in 0.XXXXs
#   [Transform] Starting...
#   [Transform] Completed in 0.XXXXs
#   [Load] Starting...
#   Loaded 100,000 records (sum: 9,999,900,000)
#   [Load] Completed in 0.XXXXs

## 3. Error Handling

try/except/finally/else — Python's exception handling. The `else` clause (runs only if no exception) is unique to Python.

In [ ]:
# WHAT: Full try/except/else/finally pattern.
# WHY: Production pipelines must handle errors gracefully —
# log the error, clean up resources, and continue or fail loudly.

import json

def parse_config(raw_json: str) -> dict:
    """Parse a JSON config string with proper error handling."""
    try:
        config = json.loads(raw_json)
    except json.JSONDecodeError as e:
        # Catch specific exception — not bare 'except:'
        print(f"  ERROR: Invalid JSON at position {e.pos}: {e.msg}")
        return {}   # return safe default
    else:
        # Runs ONLY if no exception — put success logic here
        print(f"  Parsed config with {len(config)} keys")
        return config
    finally:
        # ALWAYS runs — cleanup goes here
        print(f"  Config parsing complete")

# Valid JSON
print("Test 1 — valid:")
result = parse_config('{"model": "bert", "lr": 0.01}')

# Invalid JSON
print("\nTest 2 — invalid:")
result = parse_config('{model: broken}')

# You Should See:
# Test 1 — valid:
#   Parsed config with 2 keys
#   Config parsing complete
# Test 2 — invalid:
#   ERROR: Invalid JSON at position 1: Expecting property name enclosed in double quotes
#   Config parsing complete

In [ ]:
# WHAT: Define custom exceptions for domain-specific errors.
# WHY: 'raise ValueError' is too generic. Custom exceptions make
# error handling precise and testable.

class DataQualityError(Exception):
    """Raised when data fails quality checks."""
    def __init__(self, check_name: str, details: str):
        self.check_name = check_name
        self.details = details
        super().__init__(f"Quality check '{check_name}' failed: {details}")

class SchemaError(DataQualityError):
    """Raised when data schema doesn't match expected."""
    pass

def validate_batch(records: list[dict], required_fields: set) -> None:
    """Validate a batch of records has required fields."""
    for i, record in enumerate(records):
        missing = required_fields - set(record.keys())
        if missing:
            raise SchemaError(
                check_name="required_fields",
                details=f"Row {i} missing: {missing}"
            )

# Test with valid data
try:
    validate_batch(
        [{"id": 1, "score": 0.9}, {"id": 2}],
        required_fields={"id", "score"}
    )
except SchemaError as e:
    print(f"Caught: {e}")
    print(f"Check: {e.check_name}")

# You Should See:
# Caught: Quality check 'required_fields' failed: Row 1 missing: {'score'}
# Check: required_fields

## 4. Generators — Lazy Processing

Generators produce values one at a time with `yield`. They process gigabytes of data without loading everything into memory.

In [ ]:
# WHAT: Process a large dataset lazily with a generator.
# WHY: Loading 10GB of logs into a list crashes your machine.
# A generator processes one line at a time — constant memory.

import json
from pathlib import Path

def read_jsonl(filepath: str):
    """Generator that yields one parsed record at a time."""
    with open(filepath, "r") as f:
        for line_num, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                yield json.loads(line)
            except json.JSONDecodeError:
                print(f"  Skipping invalid JSON on line {line_num}")

# Create test data
Path("data").mkdir(exist_ok=True)
with open("data/events.jsonl", "w") as f:
    for i in range(5):
        f.write(json.dumps({"event_id": i, "type": "prediction", "score": 0.5 + i * 0.1}) + "\n")

# Process lazily — only ONE record in memory at a time
total = 0
for record in read_jsonl("data/events.jsonl"):
    total += 1
    print(f"  Processing event {record['event_id']}: score={record['score']}")

print(f"\nProcessed {total} events")

# You Should See:
#   Processing event 0: score=0.5
#   Processing event 1: score=0.6
#   Processing event 2: score=0.7
#   Processing event 3: score=0.8
#   Processing event 4: score=0.9
# Processed 5 events

In [ ]:
# WHAT: Chain generators for a multi-stage processing pipeline.
# WHY: Each stage transforms the stream without materializing
# intermediate results. This is how Spark and Kafka work conceptually.

def generate_scores(n: int):
    """Stage 1: Generate raw prediction scores."""
    import random
    random.seed(42)
    for i in range(n):
        yield {"id": i, "score": round(random.random(), 3)}

def filter_confident(records, threshold: float = 0.5):
    """Stage 2: Keep only confident predictions."""
    for r in records:
        if r["score"] >= threshold:
            yield r

def add_label(records):
    """Stage 3: Add classification label."""
    for r in records:
        yield {**r, "label": "positive"}

# Chain the pipeline — nothing executes until we iterate
pipeline = add_label(filter_confident(generate_scores(10)))

# NOW it executes — one record flows through all three stages
results = list(pipeline)
print(f"10 generated -> {len(results)} passed filter")
for r in results[:3]:
    print(f"  {r}")

# You Should See:
# 10 generated -> N passed filter
#   {'id': X, 'score': 0.XXX, 'label': 'positive'}
#   ...

## 5. Pydantic — Data Validation with Type Hints

Pydantic validates data at runtime using type hints. FastAPI uses it for request/response validation. It catches bad data BEFORE it reaches your pipeline.

In [ ]:
# WHAT: Define a Pydantic model for API input validation.
# WHY: FastAPI auto-validates request bodies using Pydantic.
# If the data doesn't match, the client gets a 422 error
# with details — no manual validation code needed.

try:
    from pydantic import BaseModel, Field, field_validator
    
    class PredictionRequest(BaseModel):
        """Validated API input for a prediction endpoint."""
        model_id: str = Field(..., min_length=1, description="Model identifier")
        features: list[float] = Field(..., min_length=1, description="Feature vector")
        threshold: float = Field(default=0.5, ge=0.0, le=1.0)
        
        @field_validator("model_id")
        @classmethod
        def model_must_be_valid(cls, v):
            valid_models = {"bert-v1", "bert-v2", "xgboost-v1"}
            if v not in valid_models:
                raise ValueError(f"Unknown model: {v}. Valid: {valid_models}")
            return v
    
    # Valid request
    req = PredictionRequest(
        model_id="bert-v2",
        features=[0.5, 1.2, -0.3, 0.8]
    )
    print(f"Valid request: {req}")
    print(f"Threshold (default): {req.threshold}")
    
    # Invalid request — bad model_id
    print("\nTesting invalid model_id:")
    try:
        bad = PredictionRequest(model_id="gpt-5", features=[1.0])
    except Exception as e:
        print(f"  Validation error: {e}")
    
    # Invalid request — threshold out of range
    print("\nTesting invalid threshold:")
    try:
        bad = PredictionRequest(model_id="bert-v2", features=[1.0], threshold=1.5)
    except Exception as e:
        print(f"  Validation error: {e}")

except ImportError:
    print("Pydantic not installed. Install with: pip install pydantic")
    print("Pydantic is used by FastAPI for automatic request validation.")

# You Should See:
# Valid request: model_id='bert-v2' features=[0.5, 1.2, -0.3, 0.8] threshold=0.5
# Threshold (default): 0.5
# Testing invalid model_id:
#   Validation error: (details about unknown model)
# Testing invalid threshold:
#   Validation error: (details about threshold > 1.0)

In [ ]:
# WHAT: Convert between Pydantic models and dicts/JSON.
# WHY: API responses, database records, and config files
# all need serialization. Pydantic handles it cleanly.

try:
    # Convert to dict and JSON
    req_dict = req.model_dump()
    req_json = req.model_dump_json(indent=2)
    
    print("As dict:")
    print(f"  {req_dict}")
    print(f"\nAs JSON:")
    print(req_json)
    
    # Create from dict (common for API responses)
    from_dict = PredictionRequest(**{"model_id": "xgboost-v1", "features": [1.0, 2.0]})
    print(f"\nFrom dict: {from_dict}")

except NameError:
    print("Skipped — Pydantic not available (see previous cell)")

# You Should See:
# As dict:
#   {'model_id': 'bert-v2', 'features': [0.5, 1.2, -0.3, 0.8], 'threshold': 0.5}
# As JSON:
# {
#   "model_id": "bert-v2",
#   "features": [0.5, 1.2, -0.3, 0.8],
#   "threshold": 0.5
# }

## 6. Regular Expressions

Regex parses structured text — log lines, error messages, timestamps, IDs. The `re` module is in the standard library.

In [ ]:
# WHAT: Parse structured log lines with regex.
# WHY: Log parsing is a core DE task. Regex extracts fields
# from semi-structured text that split() can't handle cleanly.

import re

log_lines = [
    "2026-04-04 10:00:05 ERROR [model_server] OOM: batch_size=512, memory=15.2GB",
    "2026-04-04 10:00:06 INFO  [model_server] Prediction: score=0.945, latency=12ms",
    "2026-04-04 10:00:07 WARN  [data_loader] Slow query: 2340ms on table=calls",
    "2026-04-04 10:00:08 ERROR [api_gateway] Timeout after 30s for /predict"
]

# Pattern: timestamp, level, component, message
pattern = r"(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}) (\w+)\s+\[(\w+)\] (.+)"

print("Parsed log entries:")
for line in log_lines:
    match = re.match(pattern, line)
    if match:
        timestamp, level, component, message = match.groups()
        print(f"  {level:5s} | {component:15s} | {message}")

# You Should See:
# Parsed log entries:
#   ERROR | model_server    | OOM: batch_size=512, memory=15.2GB
#   INFO  | model_server    | Prediction: score=0.945, latency=12ms
#   WARN  | data_loader     | Slow query: 2340ms on table=calls
#   ERROR | api_gateway     | Timeout after 30s for /predict

In [ ]:
# WHAT: Extract specific values with re.findall and named groups.
# WHY: Extracting metrics from log messages for monitoring dashboards.

import re

log_message = "Prediction: score=0.945, latency=12ms, model=bert-v2, batch=64"

# findall — extract all key=value pairs
pairs = re.findall(r"(\w+)=([\w.]+)", log_message)
metrics = dict(pairs)
print(f"Extracted metrics: {metrics}")

# Named groups — more readable for complex patterns
error_log = "2026-04-04 10:00:05 ERROR [model_server] OOM: batch_size=512, memory=15.2GB"
pattern = r"(?P<timestamp>[\d-]+ [\d:]+) (?P<level>\w+)\s+\[(?P<component>\w+)\] (?P<message>.+)"

match = re.match(pattern, error_log)
if match:
    parsed = match.groupdict()
    print(f"\nNamed groups: {parsed}")

# Find all ERROR lines in a batch
log_text = "\n".join([
    "10:00:05 ERROR OOM on batch 42",
    "10:00:06 INFO  Prediction complete",
    "10:00:07 ERROR Connection reset"
])

errors = re.findall(r"\d{2}:\d{2}:\d{2} ERROR (.+)", log_text)
print(f"\nError messages: {errors}")

# You Should See:
# Extracted metrics: {'score': '0.945', 'latency': '12ms', 'model': 'bert-v2', 'batch': '64'}
# Named groups: {'timestamp': '2026-04-04 10:00:05', 'level': 'ERROR', 'component': 'model_server', 'message': 'OOM: batch_size=512, memory=15.2GB'}
# Error messages: ['OOM on batch 42', 'Connection reset']

## Recap

| Pattern | When to Use | Key Example |
|---|---|---|
| Decorator | Add behavior to functions (timing, retry, auth) | `@timer`, `@retry`, `@app.route` |
| Context manager | Guarantee cleanup (files, connections, locks) | `with open()`, `with db_connection()` |
| try/except | Handle errors gracefully, not silently | Catch specific exceptions, not bare `except:` |
| Custom exceptions | Domain-specific error types | `DataQualityError`, `SchemaError` |
| Generator | Process large data without loading all into memory | `yield` one record at a time |
| Pydantic | Validate API inputs and configs at runtime | FastAPI request/response models |
| Regex | Parse structured text (logs, IDs, timestamps) | `re.match()`, `re.findall()` |

**Full explanation:** [Advanced Python Chapter](https://github.com/sunilmogadati/systems-in-production/blob/main/playbooks/python/06_Advanced.md)

**Community:** [Delivery Momentum on Skool](https://www.skool.com/deliverymomentum)